# Unidad 3 · Comparar varianzas y proporciones

Cuaderno de la base abierta de ingeniería (Análisis de Datos). Reproduce el ejemplo resuelto (mermeladas) y la pieza del caso Estampa Sur. Lee los CSV de `../datos/`. Las cifras se comparan con `verificacion/u03_varianzas_proporciones.py`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

DATOS = Path("../datos")
RES = {}

## 1. Ejemplo · ¿la dosificadora nueva es más pareja?

In [ ]:
ej = pd.read_csv(DATOS / "u03_ejemplo.csv")
a = ej[ej.dosificadora == "A"].peso_g
b = ej[ej.dosificadora == "B"].peso_g
print(ej.groupby("dosificadora").peso_g.agg(["count", "mean", "var", "std"]))

# Una varianza: ¿B cumple σ ≤ 1,5 g?  H1: σ² > 2,25
chi2 = (len(b) - 1) * b.var(ddof=1) / 1.5**2
print("chi2 =", round(chi2, 2), " p =", round(stats.chi2.sf(chi2, len(b) - 1), 3))

# Dos varianzas: H1: σB² > σA²  (F = varianza de B sobre varianza de A)
F = b.var(ddof=1) / a.var(ddof=1)
p_F = stats.f.sf(F, len(b) - 1, len(a) - 1)
print("F =", round(F, 2), " p =", round(p_F, 4))

# Levene con la mediana (Brown-Forsythe), la opción por defecto de scipy
lev = stats.levene(a, b)
print("Levene W =", round(lev.statistic, 2), " p =", round(lev.pvalue, 3))
RES.update(ej_chi2=chi2, ej_F=F, ej_p_F=p_F, ej_levene_p=lev.pvalue)

In [ ]:
se = pd.read_csv(DATOS / "u03_ejemplo_sellos.csv")
z, p = proportions_ztest(se.sello_fallado, se.frascos)   # z con la proporción combinada
print(se.assign(tasa=se.sello_fallado / se.frascos))
print("z =", round(z, 2), " p =", round(p, 3))
RES.update(ej_z=z, ej_p_z=p)

## 2. Caso Estampa Sur · ¿varían igual las dos máquinas?

In [ ]:
r = pd.read_csv(DATOS / "registro.csv")
p6 = r[r.maquina == "pulpo 6"].desplazamiento_mm
p4 = r[r.maquina == "pulpo 4"].desplazamiento_mm
F = p6.var(ddof=1) / p4.var(ddof=1)
v1, v2 = len(p6) - 1, len(p4) - 1
p_F = 2 * min(stats.f.cdf(F, v1, v2), stats.f.sf(F, v1, v2))   # bilateral
lev = stats.levene(p6, p4)
print("F =", round(F, 2), "con", (v1, v2), "gl  p =", round(p_F, 2))
print("Levene p =", round(lev.pvalue, 2))
print("Shapiro p:", round(stats.shapiro(p6).pvalue, 2), round(stats.shapiro(p4).pvalue, 2))
RES.update(caso_F=F, caso_p_F=p_F, caso_levene_p=lev.pvalue)

In [ ]:
import matplotlib.pyplot as plt
r.boxplot(column="desplazamiento_mm", by="maquina")
plt.suptitle(""); plt.title("Desplazamiento del registro (mm)"); plt.show()

## 3. Caso Estampa Sur · ¿un proveedor entrega más prendas defectuosas?

In [ ]:
d = pd.read_csv(DATOS / "defectos.csv")
d["tasa"] = d.defectuosas / d.revisadas
tabla = np.c_[d.defectuosas, d.revisadas - d.defectuosas]
chi = stats.chi2_contingency(tabla)
print(d)
print("chi2 =", round(chi.statistic, 2), " gl =", chi.dof, " p =", round(chi.pvalue, 3))
print("aportes a chi2:\n", ((tabla - chi.expected_freq) ** 2 / chi.expected_freq).round(2))
RES.update(caso_chi2=chi.statistic, caso_p_homog=chi.pvalue)

In [ ]:
# Comparaciones de a pares con Bonferroni (3 pares: cada p se multiplica por 3)
from itertools import combinations
for i, j in combinations(range(3), 2):
    z, p = proportions_ztest(d.defectuosas[[i, j]], d.revisadas[[i, j]])
    print(f"{d.proveedor[i]} vs {d.proveedor[j]}: z = {z:.2f}  p = {p:.3f}  Bonferroni = {min(1, 3 * p):.3f}")